# DiffuGPT-S: attention dynamics and head ablation

This notebook studies attention dynamics and attention-head contributions in `diffusionfamily/diffugpt-s`, which is small enough for a T4 GPU.

Outputs:
- attention entropy traces and percentile plots
- token-class/position mismatch diagnostics
- per-head residual contribution tables
- head ablation results

Runtime target: Colab/Jupyter with a T4 GPU. The notebook uses the upstream HKUNLP DiffuLLaMA/DiffuGPT inference files and the Hugging Face model `diffusionfamily/diffugpt-s`.

## 1. Install dependencies and fetch DiffuGPT helper files

Run this once at the top of a fresh runtime. If Colab asks you to restart after installs, restart and continue from this cell again.

In [ ]:
!pip -q install "torch" "transformers==4.44.2" "huggingface_hub" "safetensors" "pandas" "matplotlib" "tqdm" "accelerate"
!curl -L -o model.py https://raw.githubusercontent.com/HKUNLP/DiffuLLaMA/main/model.py
!curl -L -o attention_patch.py https://raw.githubusercontent.com/HKUNLP/DiffuLLaMA/main/attention_patch.py

## 2. Imports and configuration

The defaults are intentionally modest for a T4. Increase `DIFFUSION_STEPS`, `GEN_LEN`, or the prompt list once the first run works.

In [ ]:
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.distributions as dists
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoTokenizer

from model import DiscreteDiffusionModel, get_anneal_attn_mask, top_p_logits

MODEL_NAME = "diffusionfamily/diffugpt-s"
BASE_MODEL_NAME = "gpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

DIFFUSION_STEPS = 64
GEN_LEN = 96
LOGITS_TEMP = 0.95
TOPP_TEMP = 0.9
SHIFT = True
SEED = 42
LARGE_LEAD_THRESHOLD = 8
OUT_DIR = Path("diffugpt_s_issue_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 3. Load DiffuGPT-S

DiffuGPT-S is about 0.1B parameters, so it should fit comfortably on a T4.

In [ ]:
torch.manual_seed(SEED)
random.seed(SEED)

config = AutoConfig.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.mask_token_id is None:
    raise ValueError("DiffuGPT tokenizer is expected to define mask_token_id.")

model = DiscreteDiffusionModel.from_pretrained(
    MODEL_NAME,
    model=BASE_MODEL_NAME,
    config=config,
    tokenizer=tokenizer,
    device=DEVICE,
).to(DEVICE)
model.eval()
if DEVICE == "cuda":
    model = model.half()

print("mask_token_id:", tokenizer.mask_token_id, tokenizer.decode([tokenizer.mask_token_id]))
print("vocab_size:", model.vocab_size)

In [ ]:
def tok_text(token_id):
    return tokenizer.decode([int(token_id)], skip_special_tokens=False).replace("\n", "\\n").replace("\t", "\\t")


## 4. Define prompts

Use the `task` field to stratify reasoning vs creative prompts. Add more prompts to aggregate the plots across many sequences.

In [ ]:
PROMPTS = [
    {
        "id": "reasoning_0",
        "task": "reasoning",
        "prompt": "Question: Natalia sold clips to 48 friends in April and half as many in May. How many clips did she sell altogether? Answer step by step.\nAnswer:",
    },
    {
        "id": "reasoning_1",
        "task": "reasoning",
        "prompt": "Question: A train travels 60 miles in 1.5 hours. What is its average speed in miles per hour? Explain briefly.\nAnswer:",
    },
    {
        "id": "creative_0",
        "task": "creative",
        "prompt": "Write a short, vivid paragraph about a city waking up after rain:\n",
    },
    {
        "id": "creative_1",
        "task": "creative",
        "prompt": "Continue this story in a whimsical style: The old library only opened its hidden door when\n",
    },
]

pd.DataFrame(PROMPTS)

## 5. Diffusion generation with histories

This mirrors the upstream `generate_samples` loop, but stores:
- `xt_history`: the masked/unmasked sequence at each diffusion step before the model forward pass
- `argmax_history`: the final-layer argmax token sequence for that step
- `final_ids`: final generated sequence used as ground truth

In [ ]:
@dataclass
class HistoryResult:
    prompt_id: str
    task: str
    prompt: str
    prefix_len: int
    final_ids: list
    final_text: str
    xt_history: list
    argmax_history: list


def make_prefix_inputs(prompt, gen_len):
    prefix = [tokenizer.bos_token_id] + tokenizer.encode(prompt, add_special_tokens=False)
    if len(prefix) >= gen_len:
        prefix = prefix[: gen_len - 1]
    src_mask = [1] * len(prefix) + [0] * (gen_len - len(prefix))
    x0 = prefix + [0] * (gen_len - len(prefix))
    return {
        "input_ids": torch.tensor([x0], dtype=torch.long),
        "src_mask": torch.tensor([src_mask], dtype=torch.long),
        "prefix_len": len(prefix),
    }


def shifted_argmax_from_logits(logits, x, shift=True):
    raw = torch.argmax(logits, dim=-1)
    if shift:
        raw = torch.cat([x[:, 0:1], raw[:, :-1]], dim=1)
    return raw


@torch.inference_mode()
def generate_with_history(prompt_record):
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    inputs = make_prefix_inputs(prompt_record["prompt"], GEN_LEN)
    x = inputs["input_ids"].to(DEVICE)
    src_mask = inputs["src_mask"].bool().to(DEVICE)
    prefix_len = int(inputs["prefix_len"])

    x_embed = model.get_embeds(x)
    seq_len = x.size(1)
    batch_size = x.size(0)
    attention_mask = get_anneal_attn_mask(
        seq_len, batch_size, dtype=x_embed.dtype, device=x.device, attn_mask_ratio=1.0
    )

    maskable_mask = ~src_mask
    xt = x.masked_fill(maskable_mask, tokenizer.mask_token_id)

    xt_history = []
    argmax_history = []

    logits = model(xt, attention_mask=attention_mask)
    argmax_history.append(shifted_argmax_from_logits(logits, x, SHIFT).detach().cpu()[0].tolist())
    xt_history.append(xt.detach().cpu()[0].tolist())

    filter_logits = top_p_logits(logits / LOGITS_TEMP, p=TOPP_TEMP)
    scores = torch.log_softmax(filter_logits, dim=-1)
    x0 = dists.Categorical(logits=scores).sample()
    if SHIFT:
        x0 = torch.cat([x[:, 0:1], x0[:, :-1]], dim=1)
    x0 = xt.masked_scatter(maskable_mask, x0[maskable_mask])

    for t in range(DIFFUSION_STEPS - 1, 0, -1):
        p_to_x0 = 1 / (t + 1)
        masked_to_x0 = maskable_mask & (torch.rand_like(x0, dtype=torch.float) < p_to_x0)
        xt.masked_scatter_(masked_to_x0, x0[masked_to_x0])
        maskable_mask = maskable_mask.masked_fill(masked_to_x0, False)

        logits = model(xt, attention_mask=attention_mask)
        argmax_history.append(shifted_argmax_from_logits(logits, x, SHIFT).detach().cpu()[0].tolist())
        xt_history.append(xt.detach().cpu()[0].tolist())

        filter_logits = top_p_logits(logits / LOGITS_TEMP, p=TOPP_TEMP)
        scores = torch.log_softmax(filter_logits, dim=-1)
        x0 = dists.Categorical(logits=scores).sample()
        if SHIFT:
            x0 = torch.cat([x[:, 0:1], x0[:, :-1]], dim=1)
        x0 = xt.masked_scatter(maskable_mask, x0[maskable_mask])

    final_ids = x0.detach().cpu()[0].tolist()
    if SHIFT:
        # Upstream DiffuGPT drops the first BOS position for final display.
        display_ids = final_ids[1:]
    else:
        display_ids = final_ids

    return HistoryResult(
        prompt_id=prompt_record["id"],
        task=prompt_record["task"],
        prompt=prompt_record["prompt"],
        prefix_len=prefix_len,
        final_ids=final_ids,
        final_text=tokenizer.decode(display_ids, skip_special_tokens=True),
        xt_history=xt_history,
        argmax_history=argmax_history,
    )

## 6. Run generation

In [ ]:
histories = []
for record in tqdm(PROMPTS):
    hist = generate_with_history(record)
    histories.append(hist)
    print("\n===", hist.prompt_id, hist.task, "===")
    print(hist.final_text[:1000])

## Attention entropy: attention entropy over 1000 text sequences

This section estimates how attention entropy evolves over diffusion time for a target generated-token position at a chosen layer and head. It runs `ISSUE47_NUM_SEQUENCES = 1000` text sequences by default, then plots the mean, median, and 5th-95th percentile band across sequences.

The default target is the first generated token after the prompt (`ISSUE47_TARGET_TOKEN_OFFSET = 0`). Change `ISSUE47_LAYER`, `ISSUE47_HEAD`, or `ISSUE47_TARGET_TOKEN_OFFSET` to probe other heads/positions. On a T4, start with `ISSUE47_NUM_SEQUENCES = 20` as a smoke test, then set it back to `1000` for the full run.

In [ ]:
ISSUE47_NUM_SEQUENCES = 1000
ISSUE47_LAYER = 0
ISSUE47_HEAD = 0
ISSUE47_TARGET_TOKEN_OFFSET = 0
ISSUE47_DIFFUSION_STEPS = 64
ISSUE47_GEN_LEN = 64
ISSUE47_BATCH_SAVE_EVERY = 50

ISSUE47_PROMPT_TEMPLATES = [
    ("reasoning", "Question: {a} plus {b} equals what? Explain briefly.\nAnswer:"),
    ("reasoning", "Question: If a train goes {a} miles in {b} hours, what is its average speed? Explain briefly.\nAnswer:"),
    ("reasoning", "Question: A box has {a} red marbles and {b} blue marbles. How many marbles are there?\nAnswer:"),
    ("creative", "Write a vivid one-paragraph scene about {thing} after rain:\n"),
    ("creative", "Continue this story: The {thing} opened only when the moonlight touched\n"),
    ("creative", "Describe a strange little {thing} in a whimsical style:\n"),
]
ISSUE47_THINGS = ["city", "library", "garden", "station", "market", "harbor", "museum", "lantern", "clocktower", "courtyard"]


def build_issue47_prompts(n=ISSUE47_NUM_SEQUENCES):
    rows = []
    for i in range(int(n)):
        task, tmpl = ISSUE47_PROMPT_TEMPLATES[i % len(ISSUE47_PROMPT_TEMPLATES)]
        a = 10 + (i * 7) % 90
        b = 2 + (i * 11) % 30
        thing = ISSUE47_THINGS[i % len(ISSUE47_THINGS)]
        rows.append({
            "id": f"issue47_{i:04d}",
            "task": task,
            "prompt": tmpl.format(a=a, b=b, thing=thing),
        })
    return rows

issue47_prompts = build_issue47_prompts()
pd.DataFrame(issue47_prompts).head()

## Run attention entropy and collect entropy traces

For each sequence, this records the target token's attention entropy at every diffusion step for the selected `(layer, head)`. Entropy is normalized by `log(sequence_length)`, so values are roughly in `[0, 1]`: lower means more focused attention, higher means more diffuse attention.

In [ ]:
def forward_logits_and_attentions(input_ids, attention_mask):
    x_embed = model.get_embeds(input_ids)
    outputs = model.denoise_model(
        inputs_embeds=x_embed,
        attention_mask=attention_mask,
        output_attentions=True,
        return_dict=True,
    )
    logits = model.get_logits(outputs.last_hidden_state)
    return logits, outputs.attentions


def attention_entropy_for_target(attentions, layer, head, target_pos):
    attn = attentions[int(layer)][0, int(head), int(target_pos)].float()
    attn = torch.clamp(attn, min=0)
    denom = attn.sum().clamp_min(1e-12)
    p = attn / denom
    entropy = -(p * torch.log(p.clamp_min(1e-12))).sum()
    normalized = entropy / math.log(max(int(attn.numel()), 2))
    return float(entropy.detach().cpu()), float(normalized.detach().cpu())


def classify_token(token_id):
    text = tokenizer.decode([int(token_id)], skip_special_tokens=False)
    stripped = text.strip()
    if int(token_id) == int(tokenizer.mask_token_id):
        return "mask"
    if text in {tokenizer.bos_token or "", tokenizer.eos_token or "", tokenizer.pad_token or ""}:
        return "special"
    if text == "" or text.isspace():
        return "whitespace"
    if "\n" in text or "\r" in text:
        return "newline"
    if stripped == "":
        return "whitespace"
    if stripped.isdigit():
        return "number"
    punct_chars = ".,;:!?-()[]{}" + chr(34) + chr(39) + "`"
    if all(ch in punct_chars for ch in stripped):
        return "punctuation"
    if stripped.isalpha():
        if text.startswith(" "):
            return "word_start"
        return "word_piece"
    if any(ch.isdigit() for ch in stripped) and any(ch.isalpha() for ch in stripped):
        return "alphanumeric"
    return "mixed"


@torch.inference_mode()
def generate_issue47_history(prompt_record):
    inputs = make_prefix_inputs(prompt_record["prompt"], ISSUE47_GEN_LEN)
    x = inputs["input_ids"].to(DEVICE)
    src_mask = inputs["src_mask"].bool().to(DEVICE)
    prefix_len = int(inputs["prefix_len"])

    x_embed = model.get_embeds(x)
    seq_len = x.size(1)
    attention_mask = get_anneal_attn_mask(
        seq_len, x.size(0), dtype=x_embed.dtype, device=x.device, attn_mask_ratio=1.0
    )

    maskable_mask = ~src_mask
    xt = x.masked_fill(maskable_mask, tokenizer.mask_token_id)
    target_pos = min(prefix_len + int(ISSUE47_TARGET_TOKEN_OFFSET), ISSUE47_GEN_LEN - 1)

    entropy_rows = []
    mismatch_rows = []
    xt_history = []
    argmax_history = []

    logits, attentions = forward_logits_and_attentions(xt, attention_mask)
    ent, ent_norm = attention_entropy_for_target(attentions, ISSUE47_LAYER, ISSUE47_HEAD, target_pos)
    argmax_tokens = shifted_argmax_from_logits(logits, x, SHIFT)
    entropy_rows.append({
        "sequence_id": prompt_record["id"],
        "task": prompt_record["task"],
        "step": 0,
        "progress": 0.0,
        "layer": ISSUE47_LAYER,
        "head": ISSUE47_HEAD,
        "target_pos_abs": target_pos,
        "target_pos_rel_gen": target_pos - prefix_len,
        "attention_entropy": ent,
        "attention_entropy_norm": ent_norm,
    })
    xt_history.append(xt.detach().cpu()[0].tolist())
    argmax_history.append(argmax_tokens.detach().cpu()[0].tolist())

    filter_logits = top_p_logits(logits / LOGITS_TEMP, p=TOPP_TEMP)
    scores = torch.log_softmax(filter_logits, dim=-1)
    x0 = dists.Categorical(logits=scores).sample()
    if SHIFT:
        x0 = torch.cat([x[:, 0:1], x0[:, :-1]], dim=1)
    x0 = xt.masked_scatter(maskable_mask, x0[maskable_mask])

    for step, t in enumerate(range(ISSUE47_DIFFUSION_STEPS - 1, 0, -1), start=1):
        p_to_x0 = 1 / (t + 1)
        masked_to_x0 = maskable_mask & (torch.rand_like(x0, dtype=torch.float) < p_to_x0)
        xt.masked_scatter_(masked_to_x0, x0[masked_to_x0])
        maskable_mask = maskable_mask.masked_fill(masked_to_x0, False)

        logits, attentions = forward_logits_and_attentions(xt, attention_mask)
        ent, ent_norm = attention_entropy_for_target(attentions, ISSUE47_LAYER, ISSUE47_HEAD, target_pos)
        argmax_tokens = shifted_argmax_from_logits(logits, x, SHIFT)
        entropy_rows.append({
            "sequence_id": prompt_record["id"],
            "task": prompt_record["task"],
            "step": step,
            "progress": step / max(ISSUE47_DIFFUSION_STEPS - 1, 1),
            "layer": ISSUE47_LAYER,
            "head": ISSUE47_HEAD,
            "target_pos_abs": target_pos,
            "target_pos_rel_gen": target_pos - prefix_len,
            "attention_entropy": ent,
            "attention_entropy_norm": ent_norm,
        })
        xt_history.append(xt.detach().cpu()[0].tolist())
        argmax_history.append(argmax_tokens.detach().cpu()[0].tolist())

        filter_logits = top_p_logits(logits / LOGITS_TEMP, p=TOPP_TEMP)
        scores = torch.log_softmax(filter_logits, dim=-1)
        x0 = dists.Categorical(logits=scores).sample()
        if SHIFT:
            x0 = torch.cat([x[:, 0:1], x0[:, :-1]], dim=1)
        x0 = xt.masked_scatter(maskable_mask, x0[maskable_mask])

    final_ids = x0.detach().cpu()[0].tolist()
    gen_positions = list(range(prefix_len, ISSUE47_GEN_LEN))
    for step, (xt_row, argmax_row) in enumerate(zip(xt_history, argmax_history)):
        for pos in gen_positions:
            current_id = int(xt_row[pos])
            final_id = int(final_ids[pos])
            argmax_id = int(argmax_row[pos])
            current_class = classify_token(current_id)
            final_class = classify_token(final_id)
            argmax_class = classify_token(argmax_id)
            visible_current_differs = (
                current_id != int(tokenizer.mask_token_id) and current_id != final_id
            )
            if visible_current_differs or argmax_id != final_id or argmax_class != final_class:
                mismatch_rows.append({
                    "sequence_id": prompt_record["id"],
                    "task": prompt_record["task"],
                    "step": step,
                    "progress": step / max(ISSUE47_DIFFUSION_STEPS - 1, 1),
                    "pos_abs": pos,
                    "pos_rel_gen": pos - prefix_len,
                    "pos_norm_gen": (pos - prefix_len) / max(len(gen_positions) - 1, 1),
                    "position_bin": pd.cut(
                        [(pos - prefix_len) / max(len(gen_positions), 1)],
                        bins=[0, 0.2, 0.4, 0.6, 0.8, 1.000001],
                        labels=["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"],
                        include_lowest=True,
                    )[0],
                    "current_token_id": current_id,
                    "current_token_str": tok_text(current_id),
                    "current_class": current_class,
                    "final_token_id": final_id,
                    "final_token_str": tok_text(final_id),
                    "final_class": final_class,
                    "argmax_token_id": argmax_id,
                    "argmax_token_str": tok_text(argmax_id),
                    "argmax_class": argmax_class,
                    "visible_current_differs_from_final": visible_current_differs,
                    "argmax_differs_from_final": argmax_id != final_id,
                    "argmax_class_differs_from_final_class": argmax_class != final_class,
                })

    return entropy_rows, mismatch_rows


issue47_entropy_rows = []
issue47_mismatch_rows = []
for i, record in enumerate(tqdm(issue47_prompts, desc="Issue 47 sequences")):
    torch.manual_seed(SEED + i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED + i)
    entropy_rows, mismatch_rows = generate_issue47_history(record)
    issue47_entropy_rows.extend(entropy_rows)
    issue47_mismatch_rows.extend(mismatch_rows)

    if ISSUE47_BATCH_SAVE_EVERY and (i + 1) % ISSUE47_BATCH_SAVE_EVERY == 0:
        pd.DataFrame(issue47_entropy_rows).to_csv(OUT_DIR / "issue47_attention_entropy_partial.csv", index=False)
        pd.DataFrame(issue47_mismatch_rows).to_csv(OUT_DIR / "issue47_token_mismatches_partial.csv", index=False)

issue47_entropy_df = pd.DataFrame(issue47_entropy_rows)
issue47_mismatch_df = pd.DataFrame(issue47_mismatch_rows)
issue47_entropy_df.head(), issue47_mismatch_df.head()

## Plot attention entropy: mean, median, and 5th-95th percentile attention entropy

This is the requested 1000-sequence aggregate: the shaded band is the 5th to 95th percentile, the solid line is the mean, and the dashed line is the median.

In [ ]:
issue47_summary_df = issue47_entropy_df.groupby("step", as_index=False).agg(
    progress=("progress", "mean"),
    entropy_mean=("attention_entropy_norm", "mean"),
    entropy_median=("attention_entropy_norm", "median"),
    entropy_p05=("attention_entropy_norm", lambda s: s.quantile(0.05)),
    entropy_p95=("attention_entropy_norm", lambda s: s.quantile(0.95)),
    n_sequences=("sequence_id", "nunique"),
)

plt.figure(figsize=(9, 5))
plt.fill_between(
    issue47_summary_df["progress"],
    issue47_summary_df["entropy_p05"],
    issue47_summary_df["entropy_p95"],
    alpha=0.25,
    label="5th-95th percentile",
)
plt.plot(issue47_summary_df["progress"], issue47_summary_df["entropy_mean"], linewidth=2.2, label="mean")
plt.plot(issue47_summary_df["progress"], issue47_summary_df["entropy_median"], linestyle="--", linewidth=2.2, label="median")
plt.xlabel("Diffusion progress")
plt.ylabel("Normalized attention entropy")
plt.title(f"Issue 47 attention entropy, layer {ISSUE47_LAYER}, head {ISSUE47_HEAD}, target offset {ISSUE47_TARGET_TOKEN_OFFSET}")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "issue47_attention_entropy_percentiles.png", dpi=200)
plt.show()

issue47_summary_df.head(), issue47_summary_df.tail()

## Token classes and positions that differ from final unmasked tokens

This groups token mismatches by token class and generated-position bin. `argmax_class_differs_from_final_class` answers which classes of predicted tokens differ from their final unmasked versions; `current_differs_from_final` shows where still-masked/intermediate visible tokens differ from the final token.

In [ ]:
if len(issue47_mismatch_df):
    issue47_class_position_df = (
        issue47_mismatch_df
        .groupby(["task", "position_bin", "final_class", "argmax_class"], observed=True)
        .agg(
            rows=("sequence_id", "size"),
            sequences=("sequence_id", "nunique"),
            mean_progress=("progress", "mean"),
            mean_pos_norm=("pos_norm_gen", "mean"),
            argmax_class_mismatch_rate=("argmax_class_differs_from_final_class", "mean"),
            argmax_token_mismatch_rate=("argmax_differs_from_final", "mean"),
            visible_current_token_mismatch_rate=("visible_current_differs_from_final", "mean"),
        )
        .reset_index()
        .sort_values(["rows", "sequences"], ascending=False)
    )
else:
    issue47_class_position_df = pd.DataFrame()

print("Top class/position mismatch groups:")
display(issue47_class_position_df.head(25))

if len(issue47_mismatch_df):
    heat = issue47_mismatch_df[issue47_mismatch_df["argmax_class_differs_from_final_class"]].copy()
    pivot = pd.pivot_table(
        heat,
        values="sequence_id",
        index="final_class",
        columns="position_bin",
        aggfunc="count",
        fill_value=0,
        observed=True,
    )
    plt.figure(figsize=(10, 5))
    plt.imshow(pivot.values, aspect="auto", cmap="viridis")
    plt.colorbar(label="Mismatch rows")
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=30, ha="right")
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.xlabel("Generated position bin")
    plt.ylabel("Final token class")
    plt.title("Issue 47: argmax token-class mismatches by position")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "issue47_token_class_position_mismatches.png", dpi=200)
    plt.show()

issue47_class_position_df.head()

## Head contribution and ablation: are attention heads responsible for token prediction?

This section asks a more causal question for DiffuGPT-S: for a chosen generated token, can we identify the earliest diffusion time and residual depth where the final token is decodable, decompose the responsible layer's attention output by head, and test whether ablating high-contribution heads removes or weakens that decodability?

Important indexing note: DiffuGPT-S uses `SHIFT=True`, so the token at absolute position `p` is predicted from the residual/logits at source position `p - 1`. The analysis below uses that prediction source position.

In [ ]:
HEAD_PROBE_PROMPT_INDEX = 0
HEAD_PROBE_TOKEN_MODE = "explicit"
HEAD_PROBE_POS_REL_GEN = 0  # generated-token offset to study; change this after inspecting histories
HEAD_PROBE_TOP_ABLATE_K = 3
HEAD_PROBE_APPLY_FINAL_NORM_TO_RESIDUAL = True
HEAD_PROBE_APPLY_FINAL_NORM_TO_HEAD_DELTA = False


def diffugpt_final_norm():
    return getattr(model.denoise_model, "ln_f", None)


def diffugpt_lm_head():
    return model.lm_head


DIFFUGPT_FINAL_NORM = diffugpt_final_norm()
DIFFUGPT_LM_HEAD = diffugpt_lm_head()


def logit_lens_from_vector(vec, apply_final_norm=True):
    if vec.dim() == 1:
        vec = vec.unsqueeze(0)
    if apply_final_norm and DIFFUGPT_FINAL_NORM is not None:
        vec = DIFFUGPT_FINAL_NORM(vec)
    logits = DIFFUGPT_LM_HEAD(vec)
    return logits.squeeze(0)


def decode_top_from_vector(vec, apply_final_norm=True, topk=5):
    logits = logit_lens_from_vector(vec, apply_final_norm=apply_final_norm)
    vals, ids = torch.topk(logits, k=int(topk))
    return [
        {
            "token_id": int(tok),
            "token_str": tok_text(int(tok)),
            "logit": float(val.detach().cpu()),
        }
        for val, tok in zip(vals.detach().cpu(), ids.detach().cpu())
    ], logits


def model_hidden_states_for_ids(ids):
    input_ids = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    x_embed = model.get_embeds(input_ids)
    attention_mask = get_anneal_attn_mask(
        input_ids.size(1), input_ids.size(0), dtype=x_embed.dtype, device=input_ids.device, attn_mask_ratio=1.0
    )
    outputs = model.denoise_model(
        inputs_embeds=x_embed,
        attention_mask=attention_mask,
        output_hidden_states=True,
        return_dict=True,
    )
    logits = model.get_logits(outputs.last_hidden_state)
    return input_ids, attention_mask, logits, outputs.hidden_states


def first_unmask_time_for_final_token(xt_history, pos_abs, final_token_id):
    for t, row in enumerate(xt_history):
        tok = int(row[pos_abs])
        if tok != int(tokenizer.mask_token_id) and tok == int(final_token_id):
            return t
    return None


def select_head_probe_case():
    hist = histories[int(HEAD_PROBE_PROMPT_INDEX)]
    pos_rel = int(HEAD_PROBE_POS_REL_GEN)
    pos_abs = int(hist.prefix_len + pos_rel)
    if pos_abs >= len(hist.final_ids):
        raise ValueError(f"HEAD_PROBE_POS_REL_GEN={pos_rel} is outside final sequence length")
    source_pos_abs = max(pos_abs - 1, 0) if SHIFT else pos_abs
    final_token_id = int(hist.final_ids[pos_abs])
    unmask_time = first_unmask_time_for_final_token(hist.xt_history, pos_abs, final_token_id)
    return hist, pos_abs, pos_rel, source_pos_abs, final_token_id, unmask_time


def find_earliest_decodable_depth(hist, source_pos_abs, final_token_id, unmask_time=None):
    records = []
    best = None
    max_steps = len(hist.xt_history)
    for step, ids in enumerate(hist.xt_history):
        input_ids, attention_mask, logits, hidden_states = model_hidden_states_for_ids(ids)
        for depth_idx, hidden in enumerate(hidden_states):
            vec = hidden[0, int(source_pos_abs)]
            top, layer_logits = decode_top_from_vector(
                vec,
                apply_final_norm=HEAD_PROBE_APPLY_FINAL_NORM_TO_RESIDUAL,
                topk=1,
            )
            pred_id = int(top[0]["token_id"])
            correct = pred_id == int(final_token_id)
            transformer_layer = depth_idx - 1  # depth 0 is embeddings; depth k+1 is after layer k.
            pre_unmask = unmask_time is not None and step < int(unmask_time)
            records.append({
                "step": step,
                "progress": step / max(max_steps - 1, 1),
                "depth_idx": depth_idx,
                "transformer_layer": transformer_layer,
                "pred_token_id": pred_id,
                "pred_token_str": tok_text(pred_id),
                "final_token_id": int(final_token_id),
                "final_token_str": tok_text(final_token_id),
                "correct": correct,
                "pre_unmask": pre_unmask,
            })
            if correct and pre_unmask and best is None and transformer_layer >= 0:
                best = records[-1].copy()
                best["attention_layer_to_test"] = transformer_layer
                return best, pd.DataFrame(records)
    return best, pd.DataFrame(records)


head_probe_hist, head_probe_pos_abs, head_probe_pos_rel, head_probe_source_pos_abs, head_probe_final_token_id, head_probe_unmask_time = select_head_probe_case()
head_probe_best, head_probe_scan_df = find_earliest_decodable_depth(
    head_probe_hist,
    head_probe_source_pos_abs,
    head_probe_final_token_id,
    head_probe_unmask_time,
)

print("Selected prompt:", head_probe_hist.prompt_id, head_probe_hist.task)
print("Target token pos_abs/rel:", head_probe_pos_abs, head_probe_pos_rel)
print("Prediction source position:", head_probe_source_pos_abs)
print("Final token:", head_probe_final_token_id, tok_text(head_probe_final_token_id))
print("Unmask time:", head_probe_unmask_time)
print("Earliest pre-unmask decodable record:", head_probe_best)
head_probe_scan_df[head_probe_scan_df["correct"]].head(10)

## Per-head attention residual contributions at the first decodable layer

For the selected layer and diffusion step, this decomposes the attention output into per-head residual deltas. Each head delta is projected through the logit lens to see which tokens it pushes toward, and how much it contributes to the final token's logit.

In [ ]:
def gpt2_attention_head_contributions(block, hidden_before_block, attention_mask):
    # Mirrors GPT2Attention internals for self-attention, but keeps each head separate.
    ln_hidden = block.ln_1(hidden_before_block)
    attn = block.attn
    qkv = attn.c_attn(ln_hidden)
    query, key, value = qkv.split(attn.split_size, dim=2)
    query = attn._split_heads(query, attn.num_heads, attn.head_dim)
    key = attn._split_heads(key, attn.num_heads, attn.head_dim)
    value = attn._split_heads(value, attn.num_heads, attn.head_dim)
    attn_output, attn_weights = attn._attn(query, key, value, attention_mask, head_mask=None)
    # attn_output: [B, H, L, Dh]
    per_head = []
    for h in range(attn.num_heads):
        only_h = torch.zeros_like(attn_output)
        only_h[:, h:h+1, :, :] = attn_output[:, h:h+1, :, :]
        merged = attn._merge_heads(only_h, attn.num_heads, attn.head_dim)
        # GPT2 Conv1D c_proj is x @ W + b. For head attribution, omit shared bias.
        projected = torch.matmul(merged, attn.c_proj.weight)
        per_head.append(projected)
    per_head = torch.stack(per_head, dim=1)  # [B, H, L, D]
    bias = attn.c_proj.bias.view(1, 1, -1)
    normal_attn_delta = per_head.sum(dim=1) + bias
    return ln_hidden, per_head, normal_attn_delta, attn_weights


def run_layer_with_head_ablation(block, hidden_before_block, normal_attn_delta, per_head, ablate_heads):
    ablated_delta = normal_attn_delta.clone()
    for h in ablate_heads:
        ablated_delta = ablated_delta - per_head[:, int(h), :, :]
    post_attn = hidden_before_block + ablated_delta
    post_block = post_attn + block.mlp(block.ln_2(post_attn))
    return post_attn, post_block


if head_probe_best is None:
    raise RuntimeError("No pre-unmask decodable token found for the selected case. Try a different prompt/token.")

HEAD_PROBE_STEP = int(head_probe_best["step"])
HEAD_PROBE_LAYER = int(head_probe_best["attention_layer_to_test"])
ids_at_step = head_probe_hist.xt_history[HEAD_PROBE_STEP]
input_ids, attention_mask, logits, hidden_states = model_hidden_states_for_ids(ids_at_step)
block = model.denoise_model.h[HEAD_PROBE_LAYER]
hidden_before_block = hidden_states[HEAD_PROBE_LAYER]
ln_hidden, per_head_delta, normal_attn_delta, attn_weights = gpt2_attention_head_contributions(
    block,
    hidden_before_block,
    attention_mask,
)

source_pos = int(head_probe_source_pos_abs)
head_rows = []
for h in range(per_head_delta.shape[1]):
    delta_vec = per_head_delta[0, h, source_pos]
    top_tokens, delta_logits = decode_top_from_vector(
        delta_vec,
        apply_final_norm=HEAD_PROBE_APPLY_FINAL_NORM_TO_HEAD_DELTA,
        topk=5,
    )
    target_logit = float(delta_logits[int(head_probe_final_token_id)].detach().cpu())
    head_rows.append({
        "head": int(h),
        "target_logit_contribution": target_logit,
        "top_token_id": top_tokens[0]["token_id"],
        "top_token_str": top_tokens[0]["token_str"],
        "top_token_logit": top_tokens[0]["logit"],
        "top5_tokens": json.dumps(top_tokens, ensure_ascii=False),
    })

head_contrib_df = pd.DataFrame(head_rows).sort_values("target_logit_contribution", ascending=False)

plt.figure(figsize=(9, 4))
plt.bar(head_contrib_df["head"].astype(str), head_contrib_df["target_logit_contribution"])
plt.axhline(0, color="black", linewidth=1)
plt.xlabel(f"Attention head in layer {HEAD_PROBE_LAYER}")
plt.ylabel("Final-token logit contribution")
plt.title(f"Per-head attention contribution for final token {tok_text(head_probe_final_token_id)!r}")
plt.tight_layout()
plt.savefig(OUT_DIR / "head_probe_target_logit_contribution_by_head.png", dpi=200)
plt.show()

head_contrib_df

## Ablate high-contribution heads and test decodability

This tests whether removing the highest final-token-contributing heads at the selected layer makes the token no longer decodable at the same diffusion time/depth. It reports logit-lens predictions after the local attention sublayer and after the full transformer block.

In [ ]:
top_heads_to_ablate = head_contrib_df.head(int(HEAD_PROBE_TOP_ABLATE_K))["head"].astype(int).tolist()
print("Ablating heads:", top_heads_to_ablate)

# Normal local residuals.
normal_post_attn = hidden_before_block + normal_attn_delta
normal_post_block = normal_post_attn + block.mlp(block.ln_2(normal_post_attn))

ablation_sets = [[h] for h in top_heads_to_ablate] + [top_heads_to_ablate]
ablation_rows = []

def record_lens_state(label, vec):
    top, logits = decode_top_from_vector(
        vec,
        apply_final_norm=HEAD_PROBE_APPLY_FINAL_NORM_TO_RESIDUAL,
        topk=5,
    )
    return {
        "condition": label,
        "pred_token_id": top[0]["token_id"],
        "pred_token_str": top[0]["token_str"],
        "pred_matches_final": int(top[0]["token_id"]) == int(head_probe_final_token_id),
        "final_token_logit": float(logits[int(head_probe_final_token_id)].detach().cpu()),
        "top5_tokens": json.dumps(top, ensure_ascii=False),
    }

for point_name, tensor in [
    ("normal_after_attention", normal_post_attn),
    ("normal_after_block", normal_post_block),
]:
    ablation_rows.append({
        "ablated_heads": "none",
        "probe_point": point_name,
        **record_lens_state(point_name, tensor[0, source_pos]),
    })

for heads in ablation_sets:
    post_attn, post_block = run_layer_with_head_ablation(
        block,
        hidden_before_block,
        normal_attn_delta,
        per_head_delta,
        heads,
    )
    heads_label = ",".join(map(str, heads))
    for point_name, tensor in [
        ("ablated_after_attention", post_attn),
        ("ablated_after_block", post_block),
    ]:
        ablation_rows.append({
            "ablated_heads": heads_label,
            "probe_point": point_name,
            **record_lens_state(point_name, tensor[0, source_pos]),
        })

head_ablation_df = pd.DataFrame(ablation_rows)
head_ablation_df

## 10. Save outputs and download them

In [ ]:
# Save outputs from the attention entropy and head-ablation sections.
if "issue47_entropy_df" in globals():
    issue47_entropy_df.to_csv(OUT_DIR / "attention_entropy.csv", index=False)
if "issue47_summary_df" in globals():
    issue47_summary_df.to_csv(OUT_DIR / "attention_entropy_summary.csv", index=False)
if "issue47_mismatch_df" in globals():
    issue47_mismatch_df.to_csv(OUT_DIR / "token_mismatches.csv", index=False)
if "issue47_class_position_df" in globals():
    issue47_class_position_df.to_csv(OUT_DIR / "token_class_position_summary.csv", index=False)
if "head_probe_scan_df" in globals():
    head_probe_scan_df.to_csv(OUT_DIR / "head_probe_layer_time_scan.csv", index=False)
if "head_contrib_df" in globals():
    head_contrib_df.to_csv(OUT_DIR / "head_probe_attention_head_contributions.csv", index=False)
if "head_ablation_df" in globals():
    head_ablation_df.to_csv(OUT_DIR / "head_probe_ablation_results.csv", index=False)

summary = {
    "model_name": MODEL_NAME,
    "base_model_name": BASE_MODEL_NAME,
    "diffusion_steps": DIFFUSION_STEPS,
    "gen_len": GEN_LEN,
    "logits_temp": LOGITS_TEMP,
    "topp_temp": TOPP_TEMP,
    "shift": SHIFT,
    "seed": SEED,
    "num_prompts": len(PROMPTS),
}
if "ISSUE47_NUM_SEQUENCES" in globals():
    summary["attention_entropy"] = {
        "num_sequences": ISSUE47_NUM_SEQUENCES,
        "diffusion_steps": ISSUE47_DIFFUSION_STEPS,
        "gen_len": ISSUE47_GEN_LEN,
        "layer": ISSUE47_LAYER,
        "head": ISSUE47_HEAD,
        "target_token_offset": ISSUE47_TARGET_TOKEN_OFFSET,
    }
if "HEAD_PROBE_LAYER" in globals():
    summary["head_probe"] = {
        "prompt_index": HEAD_PROBE_PROMPT_INDEX,
        "pos_rel_gen": HEAD_PROBE_POS_REL_GEN,
        "step": HEAD_PROBE_STEP,
        "layer": HEAD_PROBE_LAYER,
        "final_token_id": head_probe_final_token_id,
        "final_token_str": tok_text(head_probe_final_token_id),
    }
with open(OUT_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

!zip -qr diffugpt_s_attention_outputs.zip diffugpt_s_issue_outputs
print("Wrote", OUT_DIR.resolve())
print("Created diffugpt_s_attention_outputs.zip")


In [ ]:
# Colab-only convenience download. If you are in local Jupyter, use the file browser instead.
try:
    from google.colab import files
    files.download("diffugpt_s_issue_outputs.zip")
except Exception as exc:
    print("Download helper skipped:", exc)